# LLM2CLIP on ShareGPT4V (SG4V) — Colab

Fine-tunes [microsoft/LLM2CLIP](https://github.com/microsoft/LLM2CLIP) on **ShareGPT4V long captions** and benchmarks on the paper's **SG4V** column (Table 1).

* Annotations: `Lin-Chen/ShareGPT4V` → `share-captioner_coco_lcs_sam_1246k_1107.json` (same file the repo's `extract_eval_embedding.py` uses for SG4V-1K).
* Images: ShareGPT4V ships no images. The COCO part (`coco/train2017`) is downloaded per image from the COCO server; LCS / SAM images are skipped (multi-GB archives).
* Validation = first `NUM_VAL` COCO entries of the file (the paper uses the first 1000 entries); training = the next `NUM_TRAIN` COCO entries.
* Training log is printed as plain text lines every `LOG_EVERY` steps (also saved to `train.log`).

**Runtime → Change runtime type → GPU**, then *Runtime → Run all*.

In [ ]:
#@title 1 · Configuration { display-mode: "form" }
# ── Model ────────────────────────────────────────────────────────────────
PRESET    = "llama3.2-1b"   #@param ["llama3.2-1b", "llama3-8b"]
INIT_FROM = "llm2clip"      #@param ["llm2clip", "eva_clip"]

# ── ShareGPT4V data ──────────────────────────────────────────────────────
SG4V_REPO = "Lin-Chen/ShareGPT4V"                          #@param {type:"string"}
SG4V_JSON = "share-captioner_coco_lcs_sam_1246k_1107.json" #@param {type:"string"}
NUM_VAL   = 1000    #@param {type:"integer"}
NUM_TRAIN = 20000   #@param {type:"integer"}
# ^ COCO images used for training. Try 2000 for a quick end-to-end test.
DL_THREADS = 32     #@param {type:"integer"}

# ── Training ─────────────────────────────────────────────────────────────
EPOCHS        = 3       #@param {type:"integer"}
BATCH_SIZE    = 0       #@param {type:"integer"}
# ^ 0 = auto from GPU memory (T4 → 32, L4 → 64, A100-40G → 128, 80G → 256)
VISUAL_LR     = 1e-5    #@param {type:"number"}
TEXT_LR       = 0.0     #@param {type:"number"}
# ^ adapter LR. 0 = auto: 1e-5 when INIT_FROM="llm2clip", 1e-4 when "eva_clip"
WEIGHT_DECAY  = 0.05    #@param {type:"number"}
VISUAL_LAYER_DECAY = 0.85  #@param {type:"number"}
WARMUP_STEPS  = 100     #@param {type:"integer"}
GRAD_CLIP     = 5.0     #@param {type:"number"}
TRAIN_LAST_N_VISION_BLOCKS = -1  #@param {type:"integer"}
# ^ -1 = whole vision tower, 0 = only vision head + adapter, N = last N blocks
GRAD_CHECKPOINTING = True  #@param {type:"boolean"}
NUM_WORKERS   = 2       #@param {type:"integer"}
LOG_EVERY     = 10      #@param {type:"integer"}
# ^ print a training log line every N steps
SEED          = 42      #@param {type:"integer"}

# ── Output ───────────────────────────────────────────────────────────────
RUN_NAME      = "llm2clip_sg4v"  #@param {type:"string"}
SAVE_TO_DRIVE = True    #@param {type:"boolean"}
# ^ mounts Google Drive: caption cache, checkpoints and logs survive a disconnect
RESUME        = True    #@param {type:"boolean"}
DOWNLOAD_BEST = True    #@param {type:"boolean"}

In [ ]:
#@title 2 · Install dependencies & clone the repo
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install timm einops ftfy regex ijson huggingface_hub
!test -d /content/LLM2CLIP || git clone -q --depth 1 https://github.com/microsoft/LLM2CLIP.git /content/LLM2CLIP
!ls /content/LLM2CLIP/llm2clip

Helper module (LLM caption encoder, model loading from the Hub, retrieval metrics).

In [ ]:
%%writefile /content/llm2clip_utils.py
"""Helpers for training microsoft/LLM2CLIP on a single GPU (Colab).

Written by the Colab notebook; also shipped inside the downloaded zip so the
best checkpoint can be loaded on your own machine.
"""
import io
import math
import os
import sys
import types

import torch
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# Presets: which LLM text encoder goes with which CLIP checkpoint / config
# ---------------------------------------------------------------------------
PRESETS = {
    # Fits a free T4 (16 GB). 2048-d caption embeddings.
    "llama3.2-1b": dict(
        llm_id="microsoft/LLM2CLIP-Llama-3.2-1B-Instruct-CC-Finetuned",
        llm_name_hint="meta-llama/Llama-3.2-1B-Instruct",
        llm_dim=2048,
        clip_cfg="Llama3.2-1B-EVA02-L-14-336",
        clip_repo="microsoft/LLM2CLIP-Llama3.2-1B-EVA02-L-14-336",
    ),
    # Paper setting. Needs bf16 + >=24 GB for the LLM (L4 / A100). 4096-d embeddings.
    "llama3-8b": dict(
        llm_id="microsoft/LLM2CLIP-Llama-3-8B-Instruct-CC-Finetuned",
        llm_name_hint="meta-llama/Meta-Llama-3-8B-Instruct",
        llm_dim=4096,
        clip_cfg="EVA02-CLIP-L-14-336",
        clip_repo="microsoft/LLM2CLIP-EVA02-L-14-336",
    ),
}
# Vision-only starting point used by the repo's run.sh (--pretrained=eva_clip)
EVA_CLIP_REPO = "QuanSun/EVA-CLIP"
EVA_CLIP_FILE = "EVA02_CLIP_L_336_psz14_s6B.pt"


# ---------------------------------------------------------------------------
# Repo import shims
# ---------------------------------------------------------------------------
def install_shims(code_dir):
    """Make microsoft/LLM2CLIP's `eva_clip` package importable on a stock Colab runtime.

    * xformers: the EVA02 configs use `xattn: true`. Installing xformers on Colab usually
      drags in a different torch build, so if it is missing we route
      `xformers.ops.memory_efficient_attention` to PyTorch's fused SDPA (same math).
      (The repo's non-xattn path also breaks under grad checkpointing, so we keep xattn on.)
    * llm2vec: `eva_clip/__init__` imports a text tower that runs the LLM online. We feed
      pre-computed LLM embeddings instead (exactly like the repo's training), so a stub
      is enough and we avoid llm2vec's pinned transformers version.
    """
    if code_dir not in sys.path:
        sys.path.insert(0, code_dir)
    os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")  # repo uses plain torch.load

    try:
        import xformers.ops  # noqa: F401
        have_xformers = not getattr(sys.modules["xformers"], "_is_shim", False)
    except Exception:
        have_xformers = False

        def memory_efficient_attention(q, k, v, attn_bias=None, p=0.0, scale=None, **_):
            # xformers layout (B, N, H, D) -> SDPA layout (B, H, N, D)
            q, k, v = (t.transpose(1, 2) for t in (q, k, v))
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias, dropout_p=p, scale=scale)
            return out.transpose(1, 2)

        xf = types.ModuleType("xformers")
        xf._is_shim = True
        xops = types.ModuleType("xformers.ops")
        xops.memory_efficient_attention = memory_efficient_attention
        xf.ops = xops
        sys.modules["xformers"] = xf
        sys.modules["xformers.ops"] = xops

    if "llm2vec" not in sys.modules:
        stub = types.ModuleType("llm2vec")

        class LLM2Vec:  # never instantiated here
            def __init__(self, *a, **k):
                raise RuntimeError("Online LLM text tower not used in this notebook.")

        stub.LLM2Vec = LLM2Vec
        sys.modules["llm2vec"] = stub
    return have_xformers


# ---------------------------------------------------------------------------
# LLM caption encoder (same output as llm2vec.LLM2Vec.encode used by the repo)
# ---------------------------------------------------------------------------
def make_bidirectional(model):
    """Same change as the Hub repo's modeling_llama_encoder.py: every attention layer
    gets is_causal=False (the mask below then only hides padding)."""
    for m in model.modules():
        if hasattr(m, "is_causal"):
            m.is_causal = False
    return model


def padding_mask_4d(attention_mask, dtype):
    """(B, L) 0/1 padding mask -> additive (B, 1, L, L) mask with NO causal triangle.
    transformers uses a ready-made 4D mask as-is, which is what makes attention bidirectional."""
    B, L = attention_mask.shape
    mask = torch.zeros(B, 1, L, L, dtype=dtype, device=attention_mask.device)
    return mask.masked_fill(attention_mask[:, None, None, :] == 0, torch.finfo(dtype).min)


def load_bidirectional_llama(model_id, dtype=torch.bfloat16):
    """Loads the LLM2CLIP caption LLM with transformers' own LlamaModel.

    The Hub repos ship custom code (LlamaEncoderModel) that imports classes removed from
    recent transformers (e.g. LlamaFlashAttention2). Its weights are plain Llama weights,
    so we load them natively and switch attention to bidirectional ourselves."""
    import transformers
    from transformers import LlamaModel

    major = int(transformers.__version__.split(".")[0])
    dtype_kw = {"dtype": dtype} if major >= 5 else {"torch_dtype": dtype}
    model, info = LlamaModel.from_pretrained(model_id, attn_implementation="sdpa",
                                             output_loading_info=True, **dtype_kw)
    missing = [k for k in info.get("missing_keys", []) if "rotary_emb" not in k]
    if missing:
        raise RuntimeError(f"{model_id}: {len(missing)} weights not found in checkpoint, e.g. {missing[:5]}")
    return make_bidirectional(model)


class LLM2CLIPTextEncoder:
    """Mean-pooled caption embeddings from an LLM2CLIP caption-contrastive LLM.

    Mirrors `LLM2Vec(model, tok, pooling_mode="mean", max_length=512).encode(captions)`
    from llm2clip/data/extract_embedding.py:
      * Llama-3 Instruct chat wrapping: <|start_header_id|>user<|end_header_id|>\\n\\n{caption}<|eot_id|>
      * left padding with the EOS token
      * mean over the caption tokens (+<|eot_id|>) only, i.e. skip_instruction=True
      * bidirectional attention, as in the Hub repo's LlamaEncoderModel
    """

    HEADER = "<|start_header_id|>user<|end_header_id|>\n\n"
    EOT = "<|eot_id|>"

    def __init__(self, model_id, device="cuda", dtype=torch.bfloat16, max_length=512,
                 model=None, tokenizer=None):
        from transformers import AutoTokenizer

        self.device = device
        self.max_length = max_length
        self.tokenizer = tokenizer or AutoTokenizer.from_pretrained(model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        if model is None:
            model = load_bidirectional_llama(model_id, dtype)
        make_bidirectional(model)
        self.model = model.to(device).eval()

    @torch.no_grad()
    def encode(self, captions, batch_size=64, progress=True, out_dtype=torch.float16):
        from tqdm.auto import tqdm

        captions = [c.strip() for c in captions]
        order = sorted(range(len(captions)), key=lambda i: -len(captions[i]))  # less padding
        out = None
        it = range(0, len(order), batch_size)
        for s in tqdm(it, desc="LLM caption embeddings", disable=not progress):
            idx = order[s:s + batch_size]
            caps = [captions[i] for i in idx]
            full = [self.HEADER + c + self.EOT for c in caps]
            enc = self.tokenizer(full, return_tensors="pt", padding=True, truncation=True,
                                 max_length=self.max_length).to(self.device)
            doc_ids = self.tokenizer([c + self.EOT for c in caps], add_special_tokens=False,
                                     truncation=True, max_length=self.max_length)["input_ids"]
            mask = padding_mask_4d(enc["attention_mask"], self.model.dtype)
            res = self.model(input_ids=enc["input_ids"], attention_mask=mask)
            hidden = res.last_hidden_state if hasattr(res, "last_hidden_state") else res[0]
            emb = torch.stack([hidden[b, -len(d):].float().mean(0) for b, d in enumerate(doc_ids)])
            if out is None:
                out = torch.empty(len(captions), emb.shape[-1], dtype=out_dtype)
            out[torch.tensor(idx)] = emb.to(out_dtype).cpu()
        return out


# ---------------------------------------------------------------------------
# CLIP model (repo's eva_clip code) + weights from the Hugging Face Hub
# ---------------------------------------------------------------------------
def _unwrap_state_dict(sd):
    for key in ("state_dict", "model", "module"):
        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
            sd = sd[key]
    if next(iter(sd)).startswith("module."):
        sd = {k[len("module."):]: v for k, v in sd.items()}
    return sd


def _first_pt_file(repo_id):
    from huggingface_hub import list_repo_files

    files = [f for f in list_repo_files(repo_id) if f.endswith((".pt", ".pth", ".bin"))]
    if not files:
        raise FileNotFoundError(f"No .pt checkpoint found in {repo_id}")
    return sorted(files, key=lambda f: (not f.endswith(".pt"), len(f)))[0]


def build_clip_model(preset, init_from="llm2clip", checkpoint_path=None):
    """Returns (model, preprocess_train, preprocess_val).

    init_from:
      "llm2clip" -> Microsoft's released LLM2CLIP weights (vision tower + LLM adapter)
      "eva_clip" -> EVA02-CLIP-L/14-336 vision weights + freshly initialised adapter (paper recipe)
      "local"    -> a state_dict you saved (e.g. best_model.pt from this notebook)
    """
    from huggingface_hub import hf_hub_download
    from eva_clip import create_model_and_transforms

    p = PRESETS[preset]
    if init_from == "eva_clip":
        ckpt = hf_hub_download(EVA_CLIP_REPO, EVA_CLIP_FILE)
        # repo loader keeps visual.*, drops text.* and visual.head (re-initialised)
        return create_model_and_transforms(p["clip_cfg"], pretrained=ckpt, force_custom_clip=True)

    model, pre_train, pre_val = create_model_and_transforms(p["clip_cfg"], force_custom_clip=True)
    if init_from == "llm2clip":
        checkpoint_path = hf_hub_download(p["clip_repo"], _first_pt_file(p["clip_repo"]))
    elif init_from != "local":
        raise ValueError(f"unknown init_from={init_from!r}")
    sd = _unwrap_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=False))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [k for k in missing if "freqs_" not in k]  # rope tables are rebuilt by the model
    if missing or unexpected:
        print(f"[warn] missing keys: {missing[:8]}{'...' if len(missing) > 8 else ''}")
        print(f"[warn] unexpected keys: {unexpected[:8]}{'...' if len(unexpected) > 8 else ''}")
    else:
        print(f"Loaded all weights from {os.path.basename(checkpoint_path)}")
    return model, pre_train, pre_val


def vision_layer_id(name, depth):
    """Layer index used for layer-wise LR decay (same grouping as training/optim.py)."""
    if name.startswith(("patch_embed", "cls_token", "pos_embed", "mask_token")):
        return 0
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1
    return depth + 1  # norm / fc_norm / head


def configure_trainable(model, train_last_n_vision_blocks=-1):
    """-1: whole vision tower trains; 0: only vision head + adapter; N: last N blocks too."""
    depth = len(model.visual.blocks)
    for name, prm in model.visual.named_parameters():
        lid = vision_layer_id(name, depth)
        if train_last_n_vision_blocks < 0:
            prm.requires_grad = True
        elif lid == depth + 1:
            prm.requires_grad = True
        elif lid == 0:
            prm.requires_grad = train_last_n_vision_blocks >= depth
        else:
            prm.requires_grad = lid > depth - train_last_n_vision_blocks
    for prm in model.text.parameters():
        prm.requires_grad = True
    model.logit_scale.requires_grad = True


def param_groups(model, visual_lr, text_lr, weight_decay, visual_layer_decay=1.0):
    depth = len(model.visual.blocks)
    groups = {}

    def add(prm, lr, wd, key):
        g = groups.setdefault(key, {"params": [], "lr": lr, "weight_decay": wd})
        g["params"].append(prm)

    skip_wd = {"pos_embed", "cls_token"}
    for name, prm in model.visual.named_parameters():
        if not prm.requires_grad:
            continue
        lid = vision_layer_id(name, depth)
        scale = visual_layer_decay ** (depth + 1 - lid)
        no_wd = prm.ndim < 2 or name.endswith(".bias") or name in skip_wd
        add(prm, visual_lr * scale, 0.0 if no_wd else weight_decay, ("v", lid, no_wd))
    for name, prm in model.text.named_parameters():
        if not prm.requires_grad:
            continue
        no_wd = prm.ndim < 2 or name.endswith(".bias")
        add(prm, text_lr, 0.0 if no_wd else weight_decay, ("t", no_wd))
    add(model.logit_scale, text_lr, 0.0, ("scale",))
    return list(groups.values())


def cosine_with_warmup(warmup_steps, total_steps):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    return f


def save_weights(model, path):
    """fp16 parameters + fp32 buffers: same format as Microsoft's released .pt files."""
    params = {n for n, _ in model.named_parameters()}
    sd = {k: (v.half() if k in params and v.is_floating_point() else v).detach().cpu()
          for k, v in model.state_dict().items()}
    torch.save(sd, path)


# ---------------------------------------------------------------------------
# Data + retrieval metrics
# ---------------------------------------------------------------------------
def load_pil(x):
    from PIL import Image

    if isinstance(x, Image.Image):
        img = x
    elif isinstance(x, dict) and x.get("bytes"):
        img = Image.open(io.BytesIO(x["bytes"]))
    elif isinstance(x, dict) and x.get("path"):
        img = Image.open(x["path"])
    elif isinstance(x, (bytes, bytearray)):
        img = Image.open(io.BytesIO(x))
    elif isinstance(x, str):
        img = Image.open(x)
    else:
        raise TypeError(f"Unsupported image type: {type(x)}")
    return img.convert("RGB")


@torch.no_grad()
def retrieval_metrics(img_feats, txt_feats, txt2img, chunk=1024):
    """img_feats (M,E), txt_feats (C,E) L2-normalised; txt2img (C,) owner image of each caption.
    Image->text uses the best-ranked of an image's captions (standard Flickr/COCO protocol)."""
    M, C = img_feats.shape[0], txt_feats.shape[0]
    dev = img_feats.device
    txt2img = txt2img.to(dev)

    # text -> image
    t2i_rank = torch.empty(C, dtype=torch.long, device=dev)
    for s in range(0, C, chunk):
        sim = txt_feats[s:s + chunk] @ img_feats.T
        gt = sim.gather(1, txt2img[s:s + chunk, None])
        t2i_rank[s:s + chunk] = (sim > gt).sum(1)

    # image -> text
    counts = torch.bincount(txt2img, minlength=M)
    kmax = int(counts.max())
    order = torch.argsort(txt2img, stable=True)
    starts = torch.cumsum(counts, 0) - counts
    slot = torch.arange(C, device=dev) - starts[txt2img[order]]
    img2txt = torch.full((M, kmax), -1, dtype=torch.long, device=dev)
    img2txt[txt2img[order], slot] = order
    i2t_rank = torch.empty(M, dtype=torch.long, device=dev)
    for s in range(0, M, chunk):
        sim = img_feats[s:s + chunk] @ txt_feats.T
        idx = img2txt[s:s + chunk]
        gt = sim.gather(1, idx.clamp(min=0)).masked_fill(idx < 0, float("-inf")).max(1, keepdim=True).values
        i2t_rank[s:s + chunk] = (sim > gt).sum(1)

    out = {}
    for k in (1, 5, 10):
        out[f"i2t_R@{k}"] = (i2t_rank < k).float().mean().item() * 100
        out[f"t2i_R@{k}"] = (t2i_rank < k).float().mean().item() * 100
    out["mean_R@1"] = (out["i2t_R@1"] + out["t2i_R@1"]) / 2
    out["rsum"] = sum(v for k, v in out.items() if "_R@" in k and k != "mean_R@1")
    return out

In [ ]:
#@title 3 · Setup (imports, GPU, output folders, optional HF login)
import os, sys, gc, json, math, time, random, hashlib, zipfile
import numpy as np, torch

os.chdir("/content")
sys.path.insert(0, "/content")
import llm2clip_utils as U

REPO_DIR = "/content/LLM2CLIP"
have_xformers = U.install_shims(os.path.join(REPO_DIR, "llm2clip"))
from eva_clip import ClipLoss  # repo code (also validates the shims)

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
device = "cuda"
gpu = torch.cuda.get_device_properties(0)
GPU_GB = gpu.total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(f"GPU: {gpu.name} ({GPU_GB:.0f} GB) | AMP: {AMP_DTYPE} | xformers: {have_xformers} (SDPA fallback otherwise)")

P = U.PRESETS[PRESET]
if PRESET == "llama3-8b" and (GPU_GB < 22 or not BF16):
    print("⚠️  The 8B LLM needs bf16 and ~22 GB for caption embedding. Use an L4/A100 or PRESET='llama3.2-1b'.")
if not BATCH_SIZE:
    BATCH_SIZE = 32 if GPU_GB < 20 else 64 if GPU_GB < 30 else 128 if GPU_GB < 50 else 256
if not TEXT_LR:
    TEXT_LR = 1e-5 if INIT_FROM == "llm2clip" else 1e-4
print(f"Batch size {BATCH_SIZE} | visual LR {VISUAL_LR} | adapter LR {TEXT_LR}")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/llm2clip_runs"
else:
    BASE_DIR = "/content/llm2clip_runs"
OUT_DIR = os.path.join(BASE_DIR, RUN_NAME)
CACHE_DIR = os.path.join(BASE_DIR, "cache")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
print("Outputs →", OUT_DIR)

# Optional: add a Colab secret named HF_TOKEN (key icon on the left) for gated/private repos or faster downloads
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
except Exception:
    _tok = os.environ.get("HF_TOKEN")
if _tok:
    from huggingface_hub import login
    login(_tok); print("Logged in to Hugging Face")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
#@title 4 · Load ShareGPT4V annotations and download the COCO images
import io, threading, ijson, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import hf_hub_download
from PIL import Image

LOG_PATH = os.path.join(OUT_DIR, "train.log")
def log(msg=""):
    print(msg, flush=True)
    with open(LOG_PATH, "a") as f:
        f.write(msg + "\n")

def hms(s):
    s = int(max(0, s)); return f"{s // 3600}:{s % 3600 // 60:02d}:{s % 60:02d}"

# ── stream the 1.2M-entry json; stop once enough COCO entries are collected ──
ann_path = hf_hub_download(SG4V_REPO, SG4V_JSON, repo_type="dataset")
log(f"annotations: {SG4V_JSON} ({os.path.getsize(ann_path) / 1e9:.2f} GB)")
need = NUM_VAL + NUM_TRAIN
coco, seen, first_1k, n_read = [], set(), {}, 0
with open(ann_path, "rb") as f:
    for it in ijson.items(f, "item", use_float=True):
        n_read += 1
        src = it["image"].split("/")[0]
        if n_read <= 1000:
            first_1k[src] = first_1k.get(src, 0) + 1
        base = os.path.basename(it["image"])
        if src == "coco" and base not in seen:
            seen.add(base)
            coco.append({"image": base, "caption": it["conversations"][1]["value"].strip()})
        if len(coco) >= need and n_read >= 1000:
            break
log(f"read {n_read:,} entries | COCO collected {len(coco):,}/{need:,}")
log(f"paper's SG4V-1K (first 1000 entries) by source: {first_1k}")

# ── download COCO train2017 images (cached; shorter side resized to 384) ──
IMG_DIR = "/content/sg4v_images"  # local disk: much faster to read than Drive (re-downloaded after a disconnect)
os.makedirs(IMG_DIR, exist_ok=True)
_local = threading.local()

def fetch(e):
    dst = os.path.join(IMG_DIR, e["image"])
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return True
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
    for attempt in range(4):
        try:
            r = _local.s.get(f"http://images.cocodataset.org/train2017/{e['image']}", timeout=30)
            r.raise_for_status()
            im = Image.open(io.BytesIO(r.content)).convert("RGB")
            w, h = im.size; s = 384 / min(w, h)
            if s < 1:
                im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
            im.save(dst + ".tmp.jpg", quality=95); os.replace(dst + ".tmp.jpg", dst)
            return True
        except Exception:
            time.sleep(1 + attempt)
    return False

todo = coco[:need]
t0, ok, done = time.time(), set(), 0
with ThreadPoolExecutor(DL_THREADS) as ex:
    futs = {ex.submit(fetch, e): e["image"] for e in todo}
    for fu in as_completed(futs):
        done += 1
        if fu.result():
            ok.add(futs[fu])
        if done % 1000 == 0 or done == len(todo):
            log(f"  images {done:,}/{len(todo):,} | failed {done - len(ok)} | {done / (time.time() - t0):.1f} img/s | {hms(time.time() - t0)}")

val_names = {e["image"] for e in coco[:NUM_VAL]}
val_entries = [e for e in todo if e["image"] in val_names and e["image"] in ok]
train_entries = [e for e in todo if e["image"] not in val_names and e["image"] in ok]
log(f"train images: {len(train_entries):,} | val images: {len(val_entries):,} (1 long caption each)")
log(f"average caption length: {np.mean([len(e['caption'].split()) for e in train_entries]):.0f} words")

import matplotlib.pyplot as plt
plt.figure(figsize=(4, 4)); plt.imshow(U.load_pil(os.path.join(IMG_DIR, train_entries[0]["image"]))); plt.axis("off"); plt.show()
print(train_entries[0]["caption"][:600], "...")

In [ ]:
#@title 5 · Embed the long captions with the LLM2CLIP LLM (frozen, cached)
texts = [e["caption"] for e in train_entries + val_entries]
key = hashlib.md5(json.dumps([PRESET, texts]).encode()).hexdigest()[:12]
cache_file = os.path.join(CACHE_DIR, f"sg4v_emb_{PRESET}_{key}.pt")
if os.path.exists(cache_file):
    emb = torch.load(cache_file)
    log(f"loaded cached embeddings: {cache_file}")
else:
    llm_dtype = torch.bfloat16 if BF16 else (torch.float32 if PRESET == "llama3.2-1b" else torch.float16)
    enc = U.LLM2CLIPTextEncoder(P["llm_id"], device=device, dtype=llm_dtype)
    t0 = time.time()
    emb = enc.encode(texts, batch_size=64 if GPU_GB > 30 else 32)  # long captions → smaller batches
    log(f"embedded {len(texts):,} captions in {hms(time.time() - t0)}")
    del enc; gc.collect(); torch.cuda.empty_cache()
    torch.save(emb, cache_file)
assert emb.shape[1] == P["llm_dim"] and torch.isfinite(emb.float()).all()
train_emb, val_emb = emb[:len(train_entries)], emb[len(train_entries):]
log(f"train_emb {tuple(train_emb.shape)} | val_emb {tuple(val_emb.shape)}")

In [ ]:
#@title 6 · Build the CLIP model (repo's eva_clip) and load weights from Hugging Face
model, preprocess_train, preprocess_val = U.build_clip_model(PRESET, INIT_FROM)
U.configure_trainable(model, TRAIN_LAST_N_VISION_BLOCKS)
model = model.to(device)
if GRAD_CHECKPOINTING:
    model.set_grad_checkpointing(True)

n_all = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{P['clip_cfg']}: {n_all/1e6:.0f}M params, {n_train/1e6:.0f}M trainable "
      f"(vision {sum(p.numel() for p in model.visual.parameters())/1e6:.0f}M, "
      f"adapter {sum(p.numel() for p in model.text.parameters())/1e6:.0f}M)")

In [ ]:
#@title 7 · Data loaders + evaluation
from torch.utils.data import Dataset, DataLoader

class SG4VDataset(Dataset):
    def __init__(self, entries, emb, transform):
        self.entries, self.emb, self.transform = entries, emb, transform
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        img = U.load_pil(os.path.join(IMG_DIR, self.entries[i]["image"]))
        return self.transform(img), self.emb[i].float()

train_set = SG4VDataset(train_entries, train_emb, preprocess_train)
val_set = SG4VDataset(val_entries, val_emb, preprocess_val)
loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kw)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE * 2, shuffle=False, **loader_kw)
assert len(train_loader) > 0, "Fewer training images than one batch: lower BATCH_SIZE or raise NUM_TRAIN"
log(f"{len(train_loader)} steps/epoch")

@torch.no_grad()
def evaluate(model):
    model.eval()
    img = []
    for x, _ in val_loader:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            img.append(model.encode_image(x.to(device, non_blocking=True), normalize=True).float())
    txt = []
    for s in range(0, len(val_emb), 1024):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            txt.append(model.encode_text(val_emb[s:s + 1024].to(device).float(), normalize=True).float())
    return U.retrieval_metrics(torch.cat(img), torch.cat(txt), torch.arange(len(val_emb)))

In [ ]:
#@title 8 · Train (console log every LOG_EVERY steps; keeps the best checkpoint by mean R@1)
optimizer = torch.optim.AdamW(
    U.param_groups(model, VISUAL_LR, TEXT_LR, WEIGHT_DECAY, VISUAL_LAYER_DECAY), betas=(0.9, 0.98), eps=1e-6)
steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, U.cosine_with_warmup(WARMUP_STEPS, total_steps))
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
loss_fn = ClipLoss()
trainable = [p for p in model.parameters() if p.requires_grad]

BEST_PATH = os.path.join(OUT_DIR, "best_model.pt")
LAST_PATH = os.path.join(OUT_DIR, "last_checkpoint.pt")
keep_last = SAVE_TO_DRIVE and RESUME
history, step_log, start_epoch, best = [], [], 0, None

def report(m, tag):
    return (f"{tag} | I2T R@1/5/10 {m['i2t_R@1']:.1f}/{m['i2t_R@5']:.1f}/{m['i2t_R@10']:.1f} | "
            f"T2I R@1/5/10 {m['t2i_R@1']:.1f}/{m['t2i_R@5']:.1f}/{m['t2i_R@10']:.1f} | mean R@1 {m['mean_R@1']:.2f}")

def maybe_save_best(m):
    global best
    if best is None or m["mean_R@1"] > best["mean_R@1"]:
        best = dict(m)
        U.save_weights(model, BEST_PATH)
        log(f"  * new best (mean R@1 {best['mean_R@1']:.2f}) -> {BEST_PATH}")

if keep_last and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    history, step_log, start_epoch, best = ck["history"], ck["step_log"], ck["epoch"], ck["best"]
    log(f"resumed after epoch {start_epoch} (best mean R@1 {best['mean_R@1']:.2f})")

if start_epoch == 0:
    m = evaluate(model); m["epoch"] = 0; history.append(m)
    log("=" * 110)
    log(report(m, "epoch 0 (before training)"))
    maybe_save_best(m)
    log("=" * 110)

t_train, done_steps = time.time(), 0
for epoch in range(start_epoch, EPOCHS):
    model.train()
    t_ep, t_win, ep_loss = time.time(), time.time(), 0.0
    w_loss = w_i2t = w_t2i = 0.0; w_n = w_imgs = 0
    for it, (images, text_emb) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        text_emb = text_emb.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            img_f, txt_f, logit_scale = model(images, text_emb)
            loss, acc = loss_fn(img_f, txt_f, logit_scale)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        gnorm = float("nan")
        if GRAD_CLIP:
            scaler.unscale_(optimizer)
            gnorm = float(torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP))
        scaler.step(optimizer); scaler.update(); scheduler.step()
        with torch.no_grad():
            model.logit_scale.clamp_(0, math.log(100))  # as in training/train.py
        done_steps += 1

        lv = loss.item(); ep_loss += lv
        w_loss += lv; w_i2t += acc["i2t"].item(); w_t2i += acc["t2i"].item(); w_n += 1; w_imgs += images.shape[0]
        if (it + 1) % LOG_EVERY == 0 or it + 1 == steps_per_epoch:
            now = time.time()
            remaining = (EPOCHS - epoch) * steps_per_epoch - (it + 1)
            step_log.append({"step": epoch * steps_per_epoch + it, "loss": w_loss / w_n,
                             "scale": logit_scale.item(), "lr": optimizer.param_groups[-1]["lr"]})
            log(f"[ep {epoch + 1}/{EPOCHS} | step {it + 1:>4}/{steps_per_epoch} | {100 * (it + 1) / steps_per_epoch:3.0f}%] "
                f"loss {w_loss / w_n:.4f} | i2t_acc {w_i2t / w_n:.2f} t2i_acc {w_t2i / w_n:.2f} | "
                f"lr {optimizer.param_groups[-1]['lr']:.2e} | scale {logit_scale.item():.1f} | gnorm {gnorm:.2f} | "
                f"{w_imgs / (now - t_win):.1f} img/s | mem {torch.cuda.max_memory_allocated() / 1e9:.1f}GB | "
                f"elapsed {hms(now - t_train)} | eta {hms((now - t_train) / done_steps * remaining)}")
            w_loss = w_i2t = w_t2i = 0.0; w_n = w_imgs = 0; t_win = time.time()

    m = evaluate(model)
    m.update(epoch=epoch + 1, train_loss=ep_loss / steps_per_epoch, minutes=(time.time() - t_ep) / 60)
    history.append(m)
    log("-" * 110)
    log(report(m, f"epoch {epoch + 1}/{EPOCHS} done") + f" | train loss {m['train_loss']:.4f} | {m['minutes']:.1f} min")
    maybe_save_best(m)
    log("-" * 110)
    json.dump({"history": history, "best": best}, open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2)
    if keep_last:
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "history": history, "step_log": step_log, "epoch": epoch + 1, "best": best}, LAST_PATH)

json.dump({"history": history, "best": best}, open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2)
log(f"\nBest: epoch {best['epoch']} | mean R@1 {best['mean_R@1']:.2f} | rsum {best['rsum']:.1f}")

In [ ]:
#@title 9 · Training curves
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if step_log:
    ax[0].plot([s["step"] for s in step_log], [s["loss"] for s in step_log], lw=1)
ax[0].set(title="train loss", xlabel="step")
ep = [h["epoch"] for h in history]
for k in ("i2t_R@1", "t2i_R@1", "mean_R@1"):
    ax[1].plot(ep, [h[k] for h in history], marker="o", label=k)
ax[1].axvline(best["epoch"], ls="--", c="gray", lw=1)
ax[1].set(title="val retrieval (dashed = best)", xlabel="epoch", ylabel="%"); ax[1].legend()
plt.tight_layout()
CURVES_PATH = os.path.join(OUT_DIR, "training_curves.png")
plt.savefig(CURVES_PATH, dpi=120); plt.show()

In [ ]:
#@title 10 · Sanity check: retrieve captions for val images with the best weights
model.load_state_dict(torch.load(BEST_PATH, map_location="cpu"))
model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
    T = model.encode_text(val_emb.to(device).float(), normalize=True).float()
    for i in random.Random(0).sample(range(len(val_set)), 3):
        x, _ = val_set[i]
        f = model.encode_image(x[None].to(device), normalize=True).float()
        top = (f @ T.T).squeeze(0).topk(3).indices.tolist()
        plt.figure(figsize=(3, 3)); plt.imshow(U.load_pil(os.path.join(IMG_DIR, val_entries[i]["image"]))); plt.axis("off"); plt.show()
        for r, t in enumerate(top, 1):
            print(f"  {r}. [{'✓' if t == i else ' '}] {val_entries[t]['caption'][:160]}...")

In [ ]:
#@title 11 · Compare with the paper (Table 1, SG4V column)
from IPython.display import display, Markdown
PAPER_SG4V = [  # R@1 I2T / T2I, ViT-L/14 @336 block (SigLIP @384)
    ("CLIP ViT-L/14-336",                86.2, 84.0),
    ("CLIP ViT-L/14-336 +LLM2CLIP-15M",  98.1, 98.4),
    ("CLIP ViT-L/14-336 +LLM2CLIP-60M",  98.9, 99.1),
    ("EVA02 ViT-L/14-336",               91.5, 89.4),
    ("EVA02 ViT-L/14-336 +LLM2CLIP-15M", 98.8, 99.2),
    ("EVA02 ViT-L/14-336 +LLM2CLIP-60M", 98.8, 99.2),
    ("SigLIP ViT-L-384",                 90.1, 90.4),
]
ref = PAPER_SG4V[5]
ours = [(f"Ours: start ({INIT_FROM}, no fine-tune)", history[0])]
if best["epoch"] != 0:
    ours.append((f"Ours: best (epoch {best['epoch']}) fine-tuned on SG4V-COCO", best))
rows = ["| Method | SG4V I2T R@1 | SG4V T2I R@1 |", "|---|---:|---:|"]
rows += [f"| Paper: {n} | {a:.1f} | {b:.1f} |" for n, a, b in PAPER_SG4V]
for label, m in ours:
    rows.append(f"| **{label}** | **{m['i2t_R@1']:.1f}** | **{m['t2i_R@1']:.1f}** |")
    rows.append(f"| *Δ vs paper EVA02 +LLM2CLIP-60M* | {m['i2t_R@1'] - ref[1]:+.1f} | {m['t2i_R@1'] - ref[2]:+.1f} |")
notes = (f"\n\nR@1 (%) on SG4V validation = first {len(val_entries)} COCO entries of `{SG4V_JSON}`. "
         f"The paper's SG4V-1K is the first 1000 entries of the same file ({first_1k.get('coco', 0)}/1000 are COCO; "
         f"the rest are LCS/SAM images not downloaded here).  \nText encoder: `{P['llm_id']}`; the paper's rows use its "
         f"Llama-3-8B text encoder.  \nTraining: {len(train_entries):,} COCO images (validation images excluded).")
md_text = "\n".join(rows) + notes
log("\n" + md_text)
display(Markdown(md_text))
open(os.path.join(OUT_DIR, "sg4v_vs_paper.md"), "w").write(md_text + "\n")

In [ ]:
#@title 12 · Package the best model and download it to your computer
zip_path = os.path.join(OUT_DIR, f"{RUN_NAME}_best.zip")
config = {k: globals()[k] for k in [
    "PRESET", "INIT_FROM", "SG4V_REPO", "SG4V_JSON", "NUM_VAL", "NUM_TRAIN", "EPOCHS", "BATCH_SIZE", "VISUAL_LR",
    "TEXT_LR", "WEIGHT_DECAY", "VISUAL_LAYER_DECAY", "WARMUP_STEPS", "TRAIN_LAST_N_VISION_BLOCKS", "SEED"]}
config.update(llm_text_encoder=P["llm_id"], clip_config=P["clip_cfg"])
with zipfile.ZipFile(zip_path, "w") as z:
    z.write(BEST_PATH, "best_model.pt", compress_type=zipfile.ZIP_STORED)
    z.writestr("metrics.json", json.dumps({"best": best, "history": history}, indent=2))
    z.writestr("config.json", json.dumps(config, indent=2))
    z.write("/content/llm2clip_utils.py", "llm2clip_utils.py")
    for extra in ("train.log", "sg4v_vs_paper.md", "training_curves.png"):
        if os.path.exists(os.path.join(OUT_DIR, extra)):
            z.write(os.path.join(OUT_DIR, extra), extra)
print(f"{zip_path}  ({os.path.getsize(zip_path)/1e9:.2f} GB) — best epoch {best['epoch']}, mean R@1 {best['mean_R@1']:.2f}")
if DOWNLOAD_BEST:
    from google.colab import files
    files.download(zip_path)  # large file: the browser download can take a few minutes